# Scikit-Learn Pipelines


In real-world machine learning workflows, raw data rarely goes directly into a model. Instead, data passes through a multi-step sequence of transformations before model fitting:

```
Raw Data ---> Imputation ---> Feature Scaling ---> Feature Selection ---> Model Fitting ---> Prediction

```

Performing these steps manually across training, validation, and testing sets is error-prone, hard to maintain, and frequently leads to **data leakage**.

Scikit-Learn's **`Pipeline`** object (`sklearn.pipeline.Pipeline`) solves these problems by assembling a sequence of data transformers and a final estimator into a single, unified Python object.

---

A Scikit-Learn pipeline enforces a strict structural requirement on its sequential steps:

1. **Intermediate Steps (Transformers):** Must implement both `.fit()` and `.transform()` methods (or `.fit_transform()`). Examples include `StandardScaler`, `SimpleImputer`, `OneHotEncoder`, and `PCA`.
2. **Final Step (Estimator):** Must implement `.fit()`. It can be a classifier (e.g., `SVM`), a regressor (e.g., `Ridge`), or another transformer. If the final step is a predictor, the pipeline exposes `.predict()` and `.score()`.

### Sequence of opereations on a pipeline object containing $N$ steps:

* **`pipeline.fit(X, y)`:**
* For steps $1$ to $N-1$: Executes `.fit_transform()` sequentially, passing the transformed output of step $k$ as the input to step $k+1$.
* For step $N$ (final estimator): Calls `.fit(X_transformed, y)`.


* **`pipeline.predict(X_test)`:**
* For steps $1$ to $N-1$: Executes `.transform()` sequentially on `X_test`.
* For step $N$: Calls `.predict(X_transformed)`.


---

### Why Pipelines?

#### Prevention of Data Leakage

* When information from outside the training dataset is used to train the model.
> feature scalers or imputers are fitted on the entire dataset (including test or validation folds) prior to splitting.

Wrapping in `Pipeline`, `scikit-learn` automatically ensures that transformers are fitted **only** on training folds during cross-validation (`cross_val_score` or `GridSearchCV`), preventing parameters like feature means ($\mu$) or standard deviations ($\sigma$) from leaking from validation sets.

#### Cleaner code

Avoid managing multiple scalers, encoders, and models separately across your codebase

### Without Pipeline (somewhat messy)

In [ ]:

X_train_imp = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imp)
model.fit(X_train_scaled, y_train)

X_test_imp = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imp)
predictions = model.predict(X_test_scaled)


NameError: name 'imputer' is not defined

### With Pipeline (Clean)

In [ ]:
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)

### Joint Hyperparameter Search

Pipelines allow tuning hyperparameters for **preprocessing steps** and **estimators** simultaneously in a single `GridSearchCV` or `RandomizedSearchCV` execution.

### Simple Production Deployment

During deployment, exporting a single object (e.g., via `joblib.dump(pipeline, 'model.pkl')`) ensures that raw production data undergoes the exact same sequence of preprocessing steps as training data without maintaining separate serialization artifacts.

---


## Demonstrations

### Basic Pipeline Construction (`Pipeline` vs. `make_pipeline`)

Scikit-learn provides two ways to build pipelines:

1. `Pipeline`: Requires explicit tuples of `('step_name', TransformerObject)`.
2. `make_pipeline`: Automatically generates step names based on class names.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline

# 1. Create synthetic data with missing values
X, y = make_classification(n_samples=200, n_features=5, random_state=42)
rng = np.random.RandomState(42)
X[rng.rand(*X.shape) < 0.1] = np.nan  # Inject 10% missing values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Syntax Method A: Standard Pipeline (Explicit Naming) ---
pipe_a = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier())
])

pipe_a.fit(X_train, y_train)
acc_a = pipe_a.score(X_test, y_test)
print(f"Pipeline A Test Accuracy: {acc_a:.4f}")

# --- Syntax Method B: make_pipeline (Shorthand) ---
pipe_b = make_pipeline(
    SimpleImputer(strategy='mean'),
    StandardScaler(),
    KNeighborsClassifier()
)

pipe_b.fit(X_train, y_train)
acc_b = pipe_b.score(X_test, y_test)
print(f"Pipeline B Test Accuracy: {acc_b:.4f}")

Pipeline A Test Accuracy: 0.8250
Pipeline B Test Accuracy: 0.8250


---

### Preventing Data Leakage in Cross-Validation

The following code illustrates how `cross_val_score` interacts safely with a `Pipeline` versus doing naive pre-scaling.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.neighbors import KNeighborsClassifier

# Injecting missing values into dataset
X, y = make_classification(n_samples=300, n_features=10, random_state=42)
X[rng.rand(*X.shape) < 0.05] = np.nan


# WRONG WAY (Data Leakage): Fitting scaler on ALL data before CV
bad_imputer = SimpleImputer(strategy='median')
bad_scaler = StandardScaler()

# Preprocessing full X before splitting creates leakage!
X_leaked = bad_scaler.fit_transform(bad_imputer.fit_transform(X))
bad_cv_scores = cross_val_score(KNeighborsClassifier(), X_leaked, y, cv=5)

print(f"Leaked CV Mean Accuracy (Optimistic/Biased): {bad_cv_scores.mean():.4f}")


# RIGHT WAY: Using Pipeline inside CV
# Correctly fits imputer and scaler ONLY on the K-1 training folds during each CV split
safe_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    KNeighborsClassifier()
)

safe_cv_scores = cross_val_score(safe_pipeline, X, y, cv=5)
print(f"Safe Pipeline CV Mean Accuracy (Unbiased):     {safe_cv_scores.mean():.4f}")

Leaked CV Mean Accuracy (Optimistic/Biased): 0.9400
Safe Pipeline CV Mean Accuracy (Unbiased):     0.9433


---

### Hyperparameter Tuning Across Transformers & Models

To refer to parameters of individual steps inside a `Pipeline` when using `GridSearchCV`, use the convention: `<step_name>__<parameter_name>` (note the **double underscore** `__`).

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge

# Generate synthetic regression data
np.random.seed(42)
X = np.random.randn(200, 8)
y = 3 * X[:, 0] - 2 * X[:, 1] + np.random.randn(200)

# Build pipeline with named steps
model_pipeline = Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler()),
    ('regressor', Ridge())
])

# Define grid using <step_name>__<parameter_name> syntax
param_grid = {
    # Tune imputer strategy
    'imputer__strategy': ['mean', 'median'],
    # Tune scaling options
    'scaler__with_mean': [True, False],
    # Tune regularization parameter alpha in Ridge
    'regressor__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]
}

# Run Grid Search over the entire pipeline
grid_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error'
)

grid_search.fit(X, y)

print("Best Parameters Found:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Score (MSE): {-grid_search.best_score_:.4f}")

Best Parameters Found:
  imputer__strategy: mean
  regressor__alpha: 1.0
  scaler__with_mean: True

Best CV Score (MSE): 0.9812


---

### Inspecting Pipeline Steps & Artifacts

After fitting a pipeline, individual step objects and fitted attributes (e.g., feature means, fitted coefficients) can be accessed using indexing or step names.

In [ ]:
# Fitting the model pipeline from Demonstration 3
model_pipeline.fit(X, y)

# Accessing step 1 (Imputer) via string key or index
imputer_step = model_pipeline.named_steps['imputer']
print("Imputer statistics (means):", imputer_step.statistics_)

# Accessing final step (Regressor) via index
final_regressor = model_pipeline[-1]
print("Ridge Coefficients:", final_regressor.coef_)

Imputer statistics (means): [ 0.00159912  0.07046241  0.05709076  0.11929184  0.06899304  0.01194121
  0.03633436 -0.04741455]
Ridge Coefficients: [ 2.59750784 -1.94094926  0.08116935  0.0157347  -0.07984385 -0.00966843
 -0.00576511  0.04867473]


---

1. **Default to Pipelines:** Wrap all machine learning workflows involving feature transformations in a `Pipeline` or `make_pipeline`.
2. **Double Underscore Syntax:** When configuring `GridSearchCV` or `RandomizedSearchCV`, target step parameters using `stepname__parametername`.
3. **Use `ColumnTransformer` for Mixed Data Types:** For heterogeneous datasets containing both categorical and numerical features, combine `ColumnTransformer` with `Pipeline` to apply different transformation chains to different feature subsets.